# [07] Final Alcohol Craving Prediction: Subject-Dependent (SD) Study

--- 
### 📊 Study Overview
Comprehensive **Subject-Dependent (SD)** machine learning sweep (RF, XGB, ET, HGB, LR) across multiple segment lengths (10s, 20s, 30s, Full) and classification tasks (2-class, 3-class).

#### **Key Enhancements (v3.1)**
- Automated export of metrics, confusion matrices, and SHAP top-25 plots.
- Fixed `KeyError` in data merging logic.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import seaborn as sns
from tqdm.auto import tqdm
import shap

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, 
    roc_auc_score, matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

REPO_ROOT = Path('..').resolve()
RUNS_DIR = REPO_ROOT / 'runs' / '07_final_report'
EXPORT_DIR = RUNS_DIR / 'results'
Q_CSV = REPO_ROOT / 'data' / 'alcohol_Qscores.csv'

for d in ['plots/cm', 'plots/shap', 'csvs']:
    os.makedirs(EXPORT_DIR / d, exist_ok=True)

if not RUNS_DIR.exists():
    print(f"⚠️ Data directory {RUNS_DIR} not found.")
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

## 1. Feature Extraction (Optional/Regeneration)

In [2]:
# =========================================================
# Feature Extraction Logic
# =========================================================
import os
import sys
sys.path.append(os.path.abspath(".."))

from src.datasets.biosignal_dataset import BioSignalDataset
import joblib
import pandas as pd
from pathlib import Path

# Use REPO_ROOT and RUNS_DIR defined in the previous cell
SPLIT_DATA_DIR = '/path/to/split_data/alcohol'
ds = BioSignalDataset(SPLIT_DATA_DIR, 'ECG', 'PPG', 'GSR')

PREP_PKL = REPO_ROOT / 'save/preprocessed_data.pkl'
if PREP_PKL.exists():
    print(f"Loading preprocessed pkl: {PREP_PKL}")
    ds.preprocessed_data = joblib.load(PREP_PKL)
    print(f"Loaded {len(ds.preprocessed_data)} subjects.")
else:
    print(f"⚠️ PREP_PKL not found at {PREP_PKL}. You may need to run preprocessing first.")

# Durations to extract
# dur_names = {None: 'FullTrial', 30.0: '30s', 20.0: '20s', 10.0: '10s'}
dur_names = {10.0: '10s'}

def extract_and_save_all():
    """
    기존에 생성된 CSV/PKL 파일이 의심되는 경우 모든 구간에 대해 특징을 재추출합니다.
    - recursive flattening 로직이 적용된 get_feature_table_long()을 사용합니다.
    """
    for dur, name in dur_names.items():
        print(f"\n>>> Extracting Phase: {name} (dur={dur})")
        ds.get_features(window_sec=dur, overlap_ratio=0.5, save_ptt_raw_csv=RUNS_DIR/f"all_raw_ptt_data_05_70_3.csv", ptt_range=(0.05,0.7))
        
        print(f"Flattening {name} features...")
        df_long = ds.get_feature_table_long()
        if df_long.empty: continue
        
        df_wide = df_long.pivot_table(
            index=['subject', 'session', 'trial'],
            columns=['channel', 'metric'],
            values='value'
        )
        df_wide.columns = [f"{c}_{m}" for c, m in df_wide.columns]
        df_final = df_wide.reset_index()
        
        csv_path = RUNS_DIR / f'features_all_{name}_05_70_3.csv'
        df_final.to_csv(csv_path, index=False)
        print(f"✅ Saved: {csv_path} ({len(df_final)} rows, {len(df_final.columns)} cols)")

# 주석을 해제하고 실행하면 모든 특징을 다시 추출하여 파일을 생성합니다.
extract_and_save_all()

Loading preprocessed pkl: <REPO_ROOT>/save/preprocessed_data.pkl


Loaded 50 subjects.

>>> Extracting Phase: 10s (dur=10.0)


  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:16<13:28, 16.50s/it]

  4%|▍         | 2/50 [00:31<12:40, 15.84s/it]

...

Error during HRV processing: All-NaN slice encountered


100%|██████████| 50/50 [12:45<00:00, 15.46s/it]

100%|██████████| 50/50 [12:45<00:00, 15.32s/it]

Saved raw PTT dataset to <REPO_ROOT>/runs/07_final_report/all_raw_ptt_data_05_70_3.csv
Flattening 10s features...


✅ Saved: <REPO_ROOT>/runs/07_final_report/features_all_10s_05_70_3.csv (15116 rows, 196 cols)


In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

EXCLUDED_SESSIONS = {
    '1_1_030_V1', '1_1_031_V1', '1_1_032_V1', '1_1_033_V1',
    '1_1_001_V1', '1_1_001_V2', '1_1_006_V2', '1_1_009_V1', '1_1_009_V2', '1_1_016_V2'
}

def analyze_and_plot_ptt(csv_path="all_raw_ptt_data.csv", out_dir="ptt_analysis_plots"):
    """
    all_raw_ptt_data.csv를 기반으로 PTT 값을 분석합니다.
    에러가 발생한 세션들을 제외하고 플로팅합니다.
    """
    if not os.path.exists(csv_path):
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return
        
    print(f"⏳ 데이터 로딩 중: {csv_path}...")
    df = pd.read_csv(csv_path)
    
    os.makedirs(out_dir, exist_ok=True)
    
    # 1. 정상 추출된 PTT 데이터만 필터링
    valid_df = df[df['status'] == 'valid'].copy()
    
    # 2. subject + session 통합 열 생성 (예: 1_1_001_V1)
    valid_df['subject_session'] = valid_df['subject'].astype(str) + '_' + valid_df['session'].astype(str)
    
    # 3. 제외할 세션 필터링
    filtered_df = valid_df[~valid_df['subject_session'].isin(EXCLUDED_SESSIONS)].copy()
    
    print(f"필터링 전 항목 수: {len(valid_df):,}")
    print(f"결측/에러 세션 필터링 후 항목 수: {len(filtered_df):,}")
    
    ordered_sub_sess = sorted(filtered_df['subject_session'].unique())
    unique_sub_sess = len(ordered_sub_sess)
    
    sns.set_theme(style="whitegrid")
    
    # ==========================================
    # Plot 1: 전체 PTT 분포 히스토그램 (01_overall_distribution.png)
    # ==========================================
    plt.figure(figsize=(10, 6))
    sns.histplot(filtered_df['ptt_ms'], bins=100, kde=True, color='blue')
    plt.title("Overall Valid PTT Distribution (Filtered)", fontsize=16)
    plt.xlabel("PTT (ms)", fontsize=12)
    plt.ylabel("Count", fontsize=12)
    out_file1 = os.path.join(out_dir, "01_overall_distribution.png")
    plt.savefig(out_file1, dpi=600, bbox_inches='tight')
    plt.close()
    
    # ==========================================
    # Plot 2: Subject + Session 별 PTT 박스플롯 (Boxplot)
    # ==========================================
    plt.figure(figsize=(max(14, unique_sub_sess * 0.4), 6))
    sns.boxplot(data=filtered_df, x='subject_session', y='ptt_ms', order=ordered_sub_sess, showfliers=True, fliersize=2, palette="Set2")
    plt.title("PTT Range and Outliers per Subject+Session (Filtered)", fontsize=16)
    plt.xlabel("Subject + Session", fontsize=12)
    plt.ylabel("PTT (ms)", fontsize=12)
    plt.xticks(rotation=90)
    out_file2 = os.path.join(out_dir, "02_subject_session_boxplot.png")
    plt.savefig(out_file2, dpi=600, bbox_inches='tight')
    plt.close()

    # ==========================================
    # Plot 3: Subject + Session 별 PTT 바이올린 플롯 (Violinplot)
    # ==========================================
    plt.figure(figsize=(max(14, unique_sub_sess * 0.4), 6))
    sns.violinplot(data=filtered_df, x='subject_session', y='ptt_ms', order=ordered_sub_sess, inner='quartile', density_norm='width', palette="Set3")
    plt.title("PTT Violin Plot per Subject+Session (Filtered)", fontsize=16)
    plt.xlabel("Subject + Session", fontsize=12)
    plt.ylabel("PTT (ms)", fontsize=12)
    plt.xticks(rotation=90)
    out_file3 = os.path.join(out_dir, "03_subject_session_violin.png")
    plt.savefig(out_file3, dpi=600, bbox_inches='tight')
    plt.close()

    print(f"✅ 분석 완료! 에러 및 결측 세션들 제외됨. 저장 경로: {out_dir}/")

if __name__ == "__main__":
    TARGET_CSV_PATH = str(RUNS_DIR / "all_raw_ptt_data_05_70_3.csv")
    OUTPUT_FOLDER = str(RUNS_DIR / "ptt_analysis_plots_05_70_3")
    
    analyze_and_plot_ptt(csv_path=TARGET_CSV_PATH, out_dir=OUTPUT_FOLDER)


⏳ 데이터 로딩 중: <REPO_ROOT>/runs/07_final_report/all_raw_ptt_data_05_70_3.csv...


필터링 전 항목 수: 163,068
결측/에러 세션 필터링 후 항목 수: 156,482


✅ 분석 완료! 에러 및 결측 세션들 제외됨. 저장 경로: <REPO_ROOT>/runs/07_final_report/ptt_analysis_plots_05_70_3/


In [4]:
import pandas as pd

df = pd.read_csv(str(REPO_ROOT / "notebooks" / "lmm_results" / "ptt_10_80" / "ptt_10_80_strict_sorted.csv"))

# Feature에 PTT가 포함된 행만 선택
ptt_df = df[df["Feature"].astype(str).str.contains("PTT", case=False, na=False)]

# 0-based 인덱스 합
rank_sum = ptt_df.index.to_series().sum()

print(ptt_df[["Feature"]])
print("PTT index 합:", rank_sum)

          Feature
29      PTT_Range
31         PTT_SD
41         PTT_CV
50        PTT_MAD
52        PTT_IQR
66       PTT_Mean
79      PTT_Prc05
112     PTT_Prc95
115    PTT_Median
116  PTT_Skewness
118       PTT_Max
123     PTT_Prc25
138       PTT_Min
145     PTT_Prc90
147     PTT_Prc75
151  PTT_Kurtosis
153     PTT_Prc10
PTT index 합: 1666


In [5]:
df = pd.read_csv(TARGET_CSV_PATH)
v = df[df["status"] == "valid"].copy()

# RR 대비 얼마나 늦은 peak를 잡았는지
v["ptt_rr_ratio"] = v["ptt_ms"] / (v["rr_sec"] * 1000)

print(v["ptt_ms"].describe([0.01, 0.05, 0.5, 0.95, 0.99]))
print(v["ptt_rr_ratio"].describe([0.01, 0.05, 0.5, 0.95, 0.99]))

# RR의 60% 넘는 beat는 의심
bad = v[v["ptt_rr_ratio"] > 0.6]
print("n_bad =", len(bad))
print(bad[["subject","session","trial","window","beat_idx","rr_sec","ptt_ms","ptt_rr_ratio"]].head(20))

count    163068.000000
mean        358.316667
std          38.329777
min          23.437262
1%          292.968750
5%          306.640625
50%         357.421875
95%         417.968750
99%         464.843750
max         621.093512
Name: ptt_ms, dtype: float64
count    163068.000000
mean          0.479766
std           0.070986
min           0.050000
1%            0.308108
5%            0.382294
50%           0.473558
95%           0.603536
99%           0.669811
max           0.914773
Name: ptt_rr_ratio, dtype: float64
n_bad = 9054
       subject session   trial       window  beat_idx    rr_sec      ptt_ms  \
4261   1_1_005      V2    end1    end1_w005       5.0  0.753906  478.515625   
5108   1_1_005      V2   high7   high7_w002      11.0  0.619141  380.859375   
5143   1_1_005      V2   high7   high7_w005      11.0  0.626953  378.906250   
6574   1_1_005      V1    low3    low3_w003       3.0  0.710938  451.171875   
6757   1_1_012      V2  start1  start1_w008      12.0  0.578125  355

In [6]:
df_long = ds.get_feature_table_long()

In [7]:
pd.DataFrame(df_long)

,subject,session,trial,channel,metric,value
0,1_1_014,V1,low3_w000,HR,ECG,69.536815
1,1_1_014,V1,low3_w000,HR,PPG,69.785747
2,1_1_014,V1,low3_w000,HRV_ECG,HRV_MeanNN,862.304688
3,1_1_014,V1,low3_w000,HRV_ECG,HRV_SDNN,36.842862
4,1_1_014,V1,low3_w000,HRV_ECG,HRV_RMSSD,45.800214
...,...,...,...,...,...,...
2634197,1_1_025,V1,high4_w007,EDA,SCR_IPI_Mean,NaN
2634198,1_1_025,V1,high4_w007,EDA,SCR_IPI_SD,NaN
2634199,1_1_025,V1,high4_w007,EDA,SCR_Peaks_Amp_Mean,0.001913
2634200,1_1_025,V1,high4_w007,EDA,SCR_Peaks_Amp_SD,0.000000


---

**Everything above this point (feature extraction) is used by every downstream notebook** (`02_abnormal_modality_sessions.ipynb` depends directly on this notebook's PTT output; `03`, `04`, and `05_ml_classification_lmm_shap.ipynb` all read the `features_all_*.csv` files written here).

**Everything below this point is a preliminary/exploratory ML sweep from before the final pipeline (`05_ml_classification_lmm_shap.ipynb`) existed.** It uses its own, simpler `load_and_label`/`run_sd_experiment` (different model roster, no abnormal-session filtering, no LMM/SHAP) and is **not** part of the paper's reported Tables III-VII — it's kept here for historical reference only. Reproduce the paper's results from `05_ml_classification_lmm_shap.ipynb`, not from the cells below.

## 2. Data Loader & Engine

In [8]:
# =========================================================
# SHAP Helpers (Integrated from Notebook 05)
# =========================================================
def get_feature_names_after_imputer(imputer, base_feature_names):
    base_names = list(base_feature_names)
    if hasattr(imputer, "indicator_") and imputer.indicator_ is not None:
        idxs = imputer.indicator_.features_
        ind_names = [f"missing__{base_names[int(i)]}" for i in idxs]
        return base_names + ind_names
    return base_names

def _unwrap_shap_to_class_arrays(shap_vals):
    # \"\"\"Robustly unwrap SHAP values to a list of class-specific arrays, handling XGBoost string bug.\"\"\"
    import re
    if hasattr(shap_vals, "values"):
        shap_vals = shap_vals.values
    
    # Check if we have an array of strings (common in some XGBoost versions)
    if isinstance(shap_vals, (list, np.ndarray)) and len(shap_vals) > 0:
        # Check first element (might be nested)
        first = shap_vals[0]
        if isinstance(first, (list, np.ndarray)) and len(first) > 0:
            first = first[0]
            
        if isinstance(first, str) and "[" in first:
            def parse_shap_obj(obj):
                if not isinstance(obj, str): return obj
                # Extract all numbers (including scientific notation)
                nums = re.findall(r"[-+]?\d*\.\d+[eE][-+]?\d+|[-+]?\d+\.\d+|[-+]?\d+", obj)
                if len(nums) == 1: return float(nums[0])
                return [float(x) for x in nums]
            
            orig_shape = getattr(shap_vals, 'shape', None)
            if isinstance(shap_vals, np.ndarray):
                # Apply parser to each element
                flat = shap_vals.ravel()
                parsed = [parse_shap_obj(s) for s in flat]
                shap_vals = np.array(parsed).reshape(orig_shape) if orig_shape else np.array(parsed)
            else:
                shap_vals = [parse_shap_obj(s) for s in shap_vals]

    if isinstance(shap_vals, np.ndarray):
        if shap_vals.ndim == 2: return [shap_vals]
        if shap_vals.ndim == 3:
            C = shap_vals.shape[2]
            return [shap_vals[:, :, ci] for ci in range(C)]
            
    if isinstance(shap_vals, list):
        out = []
        for v in shap_vals:
            if hasattr(v, "values"): v = v.values
            v = np.asarray(v)
            if v.ndim == 3 and v.shape[2] == 1: v = v[:, :, 0]
            out.append(v)
        return out
    
    return [shap_vals]
def compute_shap_values_fast(pipe, X_background_df, X_explain_df, model_name):
    scaler = pipe.named_steps['scaler']
    est = pipe.named_steps['clf']
    Xe_model = scaler.transform(X_explain_df)
    Xb_model = scaler.transform(X_background_df)
    
    if model_name in ['RF', 'ET', 'HGB', 'XGB']:
        explainer = shap.TreeExplainer(est)
        shap_vals = explainer.shap_values(Xe_model)
        return shap_vals, Xe_model
    if model_name == 'LR':
        explainer = shap.LinearExplainer(est, Xb_model)
        shap_vals = explainer.shap_values(Xe_model)
        return shap_vals, Xe_model
    return None, None

def save_shap_plot(shap_vals, X_idx, feature_names, model_name, save_prefix, classes):
    shap_list = _unwrap_shap_to_class_arrays(shap_vals)
    for i, c in enumerate(classes):
        if i >= len(shap_list): break
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_list[i], X_idx, feature_names=feature_names, show=False, max_display=25)
        plt.title(f"SHAP: {model_name} | {save_prefix} | Class {c}")
        plt.tight_layout()
        plt.savefig(EXPORT_DIR / f'plots/shap/shap_{save_prefix}_{model_name}_class{c}.png', dpi=150)
        plt.close()


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import shap
import os

SEED = 42

def load_and_label(dur_name, n_classes=2):
    csv_path = RUNS_DIR / f'features_all_{dur_name}_05_70_3.csv'  # fix: match extract_and_save_all()'s actual output filename
    if not csv_path.exists(): return None, None
    df_feat = pd.read_csv(csv_path)
    df_feat['subject'] = df_feat['subject'].astype(str).str.strip()
    df_feat['orig_trial'] = df_feat['trial'].astype(str).str.strip()
    
    # Extract base trial ID for segmented data (e.g., high1_w000 -> high1)
    df_feat['trial'] = df_feat['orig_trial'].apply(lambda x: x.split('_w')[0] if '_w' in x else x)
    
    df_q = pd.read_csv(Q_CSV)
    df_q['subject'] = df_q['subject'].astype(str).str.strip()
    df_q['version'] = df_q['version'].astype(str).str.strip()
    df_q['trial'] = df_q['trial'].astype(str).str.strip()
    
    # Align 'session' (features) with 'version' (labels)
    if 'session' in df_feat.columns and 'session' not in df_q.columns:
        df_feat = df_feat.rename(columns={'session': 'version'})
    
    if 'version' in df_feat.columns:
        df_feat['version'] = df_feat['version'].astype(str).str.strip()
        
    # Merge on subject, version, and base trial ID
    df = df_feat.merge(df_q, on=['subject', 'version', 'trial'], how='inner', suffixes=('', '_survey'))
    
    if len(df) == 0:
        return None, None

    q_vals = pd.to_numeric(df['Q_mean'], errors='coerce')
    if n_classes == 2:
        df['y'] = np.where(q_vals == 0, 0, 1)
    else:
        # 0: Zero, 1: Low (0 < Q <= 3.5), 2: High (Q > 3.5)
        df['y'] = np.where(q_vals == 0, 0, np.where(q_vals > 3.5, 2, 1))
    
    exclude = ['subject', 'version', 'trial', 'y', 'Q1', 'Q2', 'Q_mean', 'intensity', 'trial_idx', 'group_id', 'trial_0base', 'orig_trial']
    feats = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c not in exclude]
    
    # Clean Data
    df[feats] = df[feats].replace([np.inf, -np.inf], np.nan)
    df[feats] = df[feats].clip(lower=-1e15, upper=1e15)
    df[feats] = df[feats].fillna(df[feats].median()).fillna(0)
    
    # Trial-level grouping to prevent leakage
    df['group_id'] = df['subject'].astype(str) + "_" + df['version'].astype(str) + "_" + df['trial'].astype(str)
    
    return df, feats

def run_sd_experiment(df, feats, model_name='RF', save_prefix=None):
    X, y, groups = df[feats], df['y'], df['group_id']
    n_classes = len(np.unique(y))
    classes_list = sorted(np.unique(y))
    
    models = {
        'RF': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'XGB': XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
        'ET': ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'HGB': HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
        'LR': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    }
    
    clf = models[model_name]
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    if len(X) < 5:
        return {'Acc': 0, 'BAcc': 0, 'F1(Macro)': 0, 'AUROC': 0.5}
        
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    
    y_preds = np.zeros(len(y))
    y_probas = np.zeros((len(y), n_classes))
    
    # SHAP Aggregation lists (OOF samples and values)
    X_all_oof = []
    shap_all_oof = {c: [] for c in classes_list}

    fold_idx = 0
    for tr_idx, te_idx in cv.split(X, y, groups=groups):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
        
        pipe.fit(X_tr, y_tr)
        y_preds[te_idx] = pipe.predict(X_te)
        y_probas[te_idx] = pipe.predict_proba(X_te)
        
        # Compute SHAP for this fold's OOF samples
        try:
            X_bg = X_tr.sample(min(100, len(X_tr)), random_state=SEED)
            s_vals, Xe_plot = compute_shap_values_fast(pipe, X_bg, X_te, model_name)
            
            if s_vals is not None:
                X_all_oof.append(Xe_plot)
                s_list = _unwrap_shap_to_class_arrays(s_vals)
                
                # Align SHAP results with classes
                if len(s_list) == 1 and n_classes == 2:
                    # Usually SHAP for binary returns only for class 1
                    shap_all_oof[classes_list[1]].append(s_list[0])
                else:
                    for i, c in enumerate(classes_list):
                        if i < len(s_list):
                            shap_all_oof[c].append(s_list[i])
        except Exception as e:
            if fold_idx == 0:
                print(f"  [SHAP Fold Error] {model_name}: {e}")
        fold_idx += 1

    # Metrics
    res = {
        'Acc': accuracy_score(y, y_preds),
        'BAcc': balanced_accuracy_score(y, y_preds),
        'F1(Macro)': f1_score(y, y_preds, average='macro'),
    }
    if n_classes == 2:
        res['AUROC'] = roc_auc_score(y, y_probas[:, 1])
    else:
        res['AUROC'] = roc_auc_score(y, y_probas, multi_class='ovr', average='macro')
        
    if save_prefix:
        # Confusion Matrix
        try:
            conf_mat = confusion_matrix(y, y_preds)
            cm_norm = conf_mat / np.maximum(conf_mat.sum(axis=1, keepdims=True), 1e-12) * 100
            plt.figure(figsize=(6, 5))
            disp = ConfusionMatrixDisplay(confusion_matrix=conf_mat, display_labels=classes_list)
            ax = disp.plot(cmap='Blues', colorbar=False, text_kw={'color': 'none'}).ax_
            plt.title(f"{model_name} {save_prefix.replace('_',' ')}")
            plt.tight_layout()
            colormap = plt.get_cmap('Blues')
            norm_obj = colors.Normalize(vmin=conf_mat.min(), vmax=conf_mat.max())
            for i in range(conf_mat.shape[0]):
                for j in range(conf_mat.shape[1]):
                    rgba = colormap(norm_obj(conf_mat[i, j]))
                    luminance = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
                    text_color = 'black' if luminance > 0.5 else 'white'
                    ax.text(j, i, f"{conf_mat[i, j]}\n({cm_norm[i, j]:.1f}%)",
                            ha='center', va='center', color=text_color, fontsize=12, fontweight='bold')
            plt.savefig(EXPORT_DIR / f'plots/cm/cm_{save_prefix}_{model_name}.png')
            plt.close()
        except Exception as e:
            print(f"  [CM Plot Error] {e}")
            
        # Aggregated SHAP Plotting
        if X_all_oof:
            X_concat = np.concatenate(X_all_oof, axis=0)
            for c in classes_list:
                if c in shap_all_oof and shap_all_oof[c]:
                    S_concat = np.concatenate(shap_all_oof[c], axis=0)
                    if S_concat.shape[0] != X_concat.shape[0]: continue
                    
                    # 1. Summary Plot (Beeswarm)
                    plt.figure(figsize=(10, 8))
                    shap.summary_plot(S_concat, X_concat, feature_names=feats, show=False, max_display=20)
                    plt.title(f"Global SHAP (OOF): {model_name} | {save_prefix} | Class {c}")
                    plt.tight_layout()
                    plt.savefig(EXPORT_DIR / f'plots/shap/shap_{save_prefix}_{model_name}_class{c}.png', dpi=150)
                    plt.close()
                    
                    # 2. Mean Absolute Importance Plot (Bar)
                    plt.figure(figsize=(10, 8))
                    shap.summary_plot(S_concat, X_concat, feature_names=feats, show=False, max_display=20, plot_type="bar")
                    plt.title(f"Mean Abs SHAP: {model_name} | {save_prefix} | Class {c}")
                    plt.tight_layout()
                    plt.savefig(EXPORT_DIR / f'plots/shap/shap_abs_{save_prefix}_{model_name}_class{c}.png', dpi=150)
                    plt.close()

                    # 3. Export CSV
                    mean_abs = np.mean(np.abs(S_concat), axis=0)
                    imp_df = pd.DataFrame({'feature': feats, 'mean_abs_shap': mean_abs})
                    imp_df = imp_df.sort_values('mean_abs_shap', ascending=False)
                    imp_df.to_csv(EXPORT_DIR / f'csvs/shap_importance_{save_prefix}_{model_name}_class{c}.csv', index=False)
                    
    return res

In [10]:
# =========================================================
# LMM Statistical Analysis Utilities
# =========================================================

def get_modality_from_feature(feat: str) -> str:
    f = str(feat)
    if f.startswith('ECG'): return 'ECG'
    if f.startswith('PPG'): return 'PPG'
    if f.startswith('GSR'): return 'GSR'
    if f.startswith('PTT'): return 'PTT'
    return 'OTHER'

def build_lmm_table(df, feats, low_th=3.5):
    """
    Aggregates trial-level data into session×group medians.
    Requires each session to have all 3 groups (Zero, Low, High) for balanced comparison.
    """
    # Labeling groups (Zero=0, Low=0-3.5, High=3.5+)
    df['craving_group'] = 'zero'
    df.loc[(df['Q_mean'] > 0) & (df['Q_mean'] <= low_th), 'craving_group'] = 'low'
    df.loc[df['Q_mean'] > low_th, 'craving_group'] = 'high'
    
    df['craving_group'] = pd.Categorical(df['craving_group'], categories=['zero', 'low', 'high'], ordered=True)
    df['session_id'] = df['subject'].astype(str) + '_' + df['version'].astype(str)
    
    rows = []
    for (sess, grp), d in df.groupby(['session_id', 'craving_group'], observed=True):
        out = {
            'session_id': sess,
            'subject': d['subject'].iloc[0],
            'craving_group': str(grp),
            'n_trials': len(d)
        }
        for f in feats:
            vals = pd.to_numeric(d[f], errors='coerce')
            out[f] = vals.median() if not vals.isna().all() else np.nan
        rows.append(out)
    
    sdf = pd.DataFrame(rows)
    if sdf.empty: return sdf
    
    # Filter: sessions must have all 3 classes
    cnt = sdf.groupby('session_id')['craving_group'].nunique()
    keep = cnt[cnt == 3].index
    sdf = sdf[sdf['session_id'].isin(keep)].copy()
    return sdf

def run_lmm_sweep(sdf, feats, correction='fdr_bh'):
    results = []
    
    term_low  = "C(craving_group, Treatment(reference='zero'))[T.low]"
    term_high = "C(craving_group, Treatment(reference='zero'))[T.high]"
    
    for feat in feats:
        d = sdf[['subject', 'session_id', 'craving_group', feat]].dropna().copy()
        if d['subject'].nunique() < 5 or d['session_id'].nunique() < 10:
            continue
            
        try:
            # model: feature ~ group + (1|subject) + (1|session)
            model = smf.mixedlm(f"Q('{feat}') ~ C(craving_group, Treatment(reference='zero'))", 
                                data=d, groups=d['subject'], vc_formula={'session': '0 + C(session_id)'})
            fit = model.fit(reml=True, method='lbfgs', maxiter=200, disp=False)
            
            p_low = fit.pvalues.get(term_low, np.nan)
            p_high = fit.pvalues.get(term_high, np.nan)
            
            # Post-hoc: low vs high
            exog_names = fit.model.exog_names
            L = np.zeros((1, len(exog_names)))
            L[0, exog_names.index(term_low)] = 1.0
            L[0, exog_names.index(term_high)] = -1.0
            p_lh = fit.t_test(L).pvalue[0]
            
            results.append({
                'feature': feat,
                'modality': get_modality_from_feature(feat),
                'n_sessions': d['session_id'].nunique(),
                'p_zero_vs_low': p_low,
                'p_low_vs_high': p_lh,
                'p_zero_vs_high': p_high
            })
        except:
            continue
            
    res = pd.DataFrame(results)
    if res.empty: return res
    
    # FDR correction per modality
    final_rows = []
    for mod, g in res.groupby('modality'):
        ps = g[['p_zero_vs_low', 'p_low_vs_high', 'p_zero_vs_high']].values.flatten()
        finite = np.isfinite(ps)
        adj = np.full_like(ps, np.nan)
        if finite.any():
            adj[finite] = multipletests(ps[finite], method=correction)[1]
        
        n = len(g)
        g = g.copy()
        g['p_adj_zero_vs_low'] = adj[0:n]
        g['p_adj_low_vs_high'] = adj[n:2*n]
        g['p_adj_zero_vs_high'] = adj[2*n:3*n]
        final_rows.append(g)
        
    return pd.concat(final_rows).sort_values('p_adj_zero_vs_high')


In [11]:
len(load_and_label('10s', n_classes = 2)[1])

193

In [12]:
df, feats = load_and_label('10s', n_classes=2)
print(f"Running: {2}-class | {'10s'} | {'XGB'}")
r = run_sd_experiment(df, feats, model_name='XGB', save_prefix=f"{2}class_{'10s'}")

Running: 2-class | 10s | XGB


  [SHAP Fold Error] XGB: could not convert string to float: '[6.519084E-1]'


<Figure size 600x500 with 0 Axes>

In [13]:
r

{'Acc': 0.7760623662488536,
 'BAcc': 0.7322938355594273,
 'F1(Macro)': 0.7413702283307303,
 'AUROC': 0.8280544689192405}

## 2. Experimental Sweep
Sweeping across segment lengths, models, and classification tasks.

In [14]:
# durations = ['FullTrial', '30s', '20s', '10s']
durations = ['10s',]

models_list = ['RF', 'XGB', 'HGB', 'LR']
sweep_results = []

for nc in [2, 3]:
    for dur in durations:
        df, feats = load_and_label(dur, n_classes=nc)
        if df is None: continue
        for model_name in models_list:
            print(f"Running: {nc}-class | {dur} | {model_name}")
            r = run_sd_experiment(df, feats, model_name=model_name, save_prefix=f"{nc}class_{dur}")
            r.update({'Classes': nc, 'Duration': dur, 'Model': model_name})
            sweep_results.append(r)

res_df = pd.DataFrame(sweep_results)
res_df.to_csv(EXPORT_DIR / 'csvs/all_metrics_report.csv', index=False)
display(res_df.sort_values('AUROC', ascending=False).head(20))

Running: 2-class | 10s | RF


Running: 2-class | 10s | XGB


  [SHAP Fold Error] XGB: could not convert string to float: '[6.519084E-1]'


Running: 2-class | 10s | HGB


Running: 2-class | 10s | LR


...

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

In [15]:
temp = res_df.sort_values('BAcc', ascending=False)

In [16]:
temp2s = temp[temp['Classes']==2]

In [17]:
temp2s

,Acc,BAcc,F1(Macro),AUROC,Classes,Duration,Model
0,0.794329,0.738779,0.753462,0.848938,2,10s,RF
2,0.787603,0.738199,0.750268,0.835654,2,10s,HGB
1,0.776062,0.732294,0.741370,0.828054,2,10s,XGB
3,0.592556,0.600489,0.582455,0.644138,2,10s,LR


In [18]:
display(temp2s[temp2s['Duration']=='FullTrial'])

,Acc,BAcc,F1(Macro),AUROC,Classes,Duration,Model


In [19]:
# =========================================================
# Step 6: LMM Statistical Validation Sweep
# =========================================================
print("Starting LMM Statistical Sweep...")
durations = ['FullTrial', '30s', '20s', '10s']
for dur in durations:
    print(f"--- LMM Analysis: {dur} ---")
    df_lmm, feats_lmm = load_and_label(dur, n_classes=3)
    if df_lmm is None: continue
    
    sdf = build_lmm_table(df_lmm, feats_lmm)
    print(f"Sessions with all 3 classes: {sdf['session_id'].nunique()}")
    
    if sdf['session_id'].nunique() > 5:
        lmm_res = run_lmm_sweep(sdf, feats_lmm)
        out_path = EXPORT_DIR / f'csvs/lmm_report_{dur}.csv'
        lmm_res.to_csv(out_path, index=False)
        print(f"Saved LMM results to: {out_path}")
        display(lmm_res.head(10))
    else:
        print("Insufficient data for LMM sweep.")


Starting LMM Statistical Sweep...
--- LMM Analysis: FullTrial ---
--- LMM Analysis: 30s ---
--- LMM Analysis: 20s ---
--- LMM Analysis: 10s ---


Sessions with all 3 classes: 49


Saved LMM results to: <REPO_ROOT>/runs/07_final_report/results/csvs/lmm_report_10s.csv


""
